<a href="https://colab.research.google.com/github/srivatsa3809/machine-learning-specialization/blob/main/Simple_Neural_Network.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# =====================================================================
# CELL 1: DOWNLOAD & VERIFY FILE EXISTENCE
# =====================================================================
import sys
import os

# 1. Purge any cached paths or broken partial files
!rm -f lab_utils_common.py lab_coffee_utils.py deeplearning.mplstyle coffee.csv
sys.modules.pop('lab_utils_common', None)
sys.modules.pop('lab_coffee_utils', None)

print("⏳ Downloading Course 2 Week 1 Coffee Lab dependencies...")

# 2. Download utilities and style sheets directly from a complete Coursera source
!wget -q https://raw.githubusercontent.com/greyhatguy007/machine-learning-specialization-coursera/main/C2%20-%20Advanced%20Learning%20Algorithms/week1/optional-labs/lab_utils_common.py
!wget -q https://raw.githubusercontent.com/greyhatguy007/machine-learning-specialization-coursera/main/C2%20-%20Advanced%20Learning%20Algorithms/week1/optional-labs/lab_coffee_utils.py
!wget -q https://raw.githubusercontent.com/greyhatguy007/machine-learning-specialization-coursera/main/C2%20-%20Advanced%20Learning%20Algorithms/week1/optional-labs/deeplearning.mplstyle

# 3. Explicitly pull down the required coffee dataset file
!mkdir -p data
!wget -q -O data/coffee.csv https://raw.githubusercontent.com/greyhatguy007/machine-learning-specialization-coursera/main/C2%20-%20Advanced%20Learning%20Algorithms/week1/optional-labs/data/coffee.csv
# Also make a copy in root folder just in case the helper file looks locally
!cp data/coffee.csv ./coffee.csv

# 4. Inject current working directory into Python's search path manually
if os.getcwd() not in sys.path:
    sys.path.append(os.getcwd())

# 5. Enable the interactive backend layout
from google.colab import output
output.enable_custom_widget_manager()
!pip install -q ipympl

print("\n📁 Current workspace files:")
print(os.listdir('.'))
print("\n✅ Verification complete. If you see 'lab_utils_common.py' listed above, proceed to Cell 2!")

In [ ]:
# =====================================================================
# CELL 2: RUN IMPORTS & MATPLOTLIB SIDEWAYS BACKEND
# =====================================================================
# Switch plotting engine to handle slider widgets smoothly
%matplotlib widget

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

# Local Coursera lab imports
from lab_utils_common import dlc
from lab_coffee_utils import load_coffee_data, plt_roast, plt_prob, plt_layer, plt_network, plt_output_unit

# Apply the custom stylesheet
plt.style.use('./deeplearning.mplstyle')

# Silence TensorFlow verbose warnings to keep Colab outputs clean
import logging
logging.getLogger("tensorflow").setLevel(logging.ERROR)
tf.autograph.set_verbosity(0)

print("☕ Success! Course 2 Week 1 Coffee Roasting lab imports are fully functional.")

In [ ]:
X,Y = load_coffee_data();
print(X.shape, Y.shape)

In [ ]:
plt_roast(X,Y)

In [ ]:
print(f"Temperature Max, Min pre normalization: {np.max(X[:,0]):0.2f}, {np.min(X[:,0]):0.2f}")
print(f"Duration    Max, Min pre normalization: {np.max(X[:,1]):0.2f}, {np.min(X[:,1]):0.2f}")
norm_l = tf.keras.layers.Normalization(axis=-1)
norm_l.adapt(X)  # learns mean, variance
Xn = norm_l(X)
print(f"Temperature Max, Min post normalization: {np.max(Xn[:,0]):0.2f}, {np.min(Xn[:,0]):0.2f}")
print(f"Duration    Max, Min post normalization: {np.max(Xn[:,1]):0.2f}, {np.min(Xn[:,1]):0.2f}")

In [ ]:
# Define the activation function
# Define the sigmoid activation function manually
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

# Now your line will work perfectly
g = sigmoid

In [ ]:
def my_dense(a_in, W, b):
    """
    Computes dense layer
    Args:
      a_in (ndarray (n, )) : Data, 1 example
      W    (ndarray (n,j)) : Weight matrix, n features per unit, j units
      b    (ndarray (j, )) : bias vector, j units
    Returns
      a_out (ndarray (j,))  : j units|
    """
    units = W.shape[1]
    a_out = np.zeros(units)
    for j in range(units):
        w = W[:,j]
        z = np.dot(w, a_in) + b[j]
        a_out[j] = g(z)
    return(a_out)

In [ ]:
def my_sequential(x, W1, b1, W2, b2):
    a1 = my_dense(x,  W1, b1)
    a2 = my_dense(a1, W2, b2)
    return(a2)

In [ ]:
W1_tmp = np.array( [[-8.93,  0.29, 12.9 ], [-0.1,  -7.32, 10.81]] )
b1_tmp = np.array( [-9.82, -9.28,  0.96] )
W2_tmp = np.array( [[-31.18], [-27.59], [-32.56]] )
b2_tmp = np.array( [15.41] )

In [ ]:
def my_predict(X, W1, b1, W2, b2):
    m = X.shape[0]
    p = np.zeros((m,1))
    for i in range(m):
        p[i,0] = my_sequential(X[i], W1, b1, W2, b2)
    return(p)

In [ ]:
X_tst = np.array([
    [200,13.9],  # postive example
    [200,17]])   # negative example
X_tstn = norm_l(X_tst)  # remember to normalize
predictions = my_predict(X_tstn, W1_tmp, b1_tmp, W2_tmp, b2_tmp)

In [ ]:
yhat = np.zeros_like(predictions)
for i in range(len(predictions)):
    if predictions[i] >= 0.5:
        yhat[i] = 1
    else:
        yhat[i] = 0
print(f"decisions = \n{yhat}")

In [ ]:
yhat = (predictions >= 0.5).astype(int)
print(f"decisions = \n{yhat}")

In [ ]:
netf= lambda x : my_predict(norm_l(x),W1_tmp, b1_tmp, W2_tmp, b2_tmp)
plt_network(X,Y,netf)